# Benchmark (1): scoring the decision process

Run the `Llama 3 Agentic ACE` pipeline with GPT-4o for two months after its training
cutoff (decision dates **2024-07-31** and **2024-08-31**). Give the whole chat history ("chain of thought"), the prompts
and the data to another LLM (Claude to prevent self-preference bias), and ask: how well did GPT-4o arrive at its scoring rule? Did it use the appropriate tools, and did the tool use
help? Score a rubric 0–2 with evidence. One judge run per month.

**Design**
| Step | What | Code |
|---|---|---|
| 1 | Run the original notebook code **as written**. Only the LLM is swapped (`ollama/llama3` → `gpt-4o-2024-08-06`, temperature 0), and the tool rate limit is raised. Record every LLM call, tool call and task from CrewAI's event bus. | `benchmarks/b1_ace_pipeline.py` |
| 2 | **Deterministic audit** from the log: tool counts and errors, look-ahead, whether forward returns were ever used, thresholds copied from prompt examples, and whether the final rule executes on the decision date. | `benchmarks/b1_judge.py::audit` |
| 3 | **LLM judge** (Claude Opus 5.5, a different model family from the system judged). It gets the prompts, data description, audit, final output and full transcript, plus a read-only SQL tool on the same data, with returns at or after the decision date **masked**, so it grades the process blind to the outcome. | `benchmarks/b1_judge.py::judge` |
| 4 | **Human check**: reviewer columns next to each judge score. | this notebook |
| 5 | **Outcome, revealed last**: realized next-month return of the BUY/SELL lists, kept out of the rubric (Prof. Capponi: keep Sharpe out of the rubric so the link can be tested). | `benchmarks/b1_judge.py::outcome` |

**How to run:** kernel *agentic-screen (Python 3.11)*, then *Run All*. By default it loads saved runs from
`data_local/benchmarks/b1/` (git-ignored: transcripts contain licensed data). Set the flags below to re-run any LLM step.

In [ ]:
import sys, json, warnings
from pathlib import Path
import numpy as np, pandas as pd
from IPython.display import display, Markdown
warnings.filterwarnings("ignore")
HERE = Path.cwd().resolve()
REPO = next(p for p in [HERE, *HERE.parents] if (p / "xilin" / "benchmarks").exists())   # code/Agentics
sys.path.insert(0, str(REPO / "xilin"))
pd.set_option("display.width", 220, "display.max_columns", 30, "display.max_colwidth", 140, "display.precision", 4)

from benchmarks.b1_ace_pipeline import run_month
from benchmarks.b1_judge import RUBRIC, load_trajectory, render, audit, judge, outcome, B1
from benchmarks.common import cost_usd, RESULTS

MONTHS = {"2024-07": "2024-07-31", "2024-08": "2024-08-31"}
RUN_PIPELINE = False   # True: re-run GPT-4o (about 2 min and ~$0.20 per month)
RUN_JUDGE = False      # True: re-run the Claude judge (a few minutes and ~$1-3 per month)

## 1. Run the system (GPT-4o in original pipeline)

In [ ]:
for m, d in MONTHS.items():
    if RUN_PIPELINE:
        print("running", d); run_month(d)
    elif not (B1 / m / "meta.json").exists():
        FLAG = "RUN_PIPELINE"
        raise FileNotFoundError(f"No saved run in {B1 / m}. Unzip the shared benchmark runs into "
                                f"data_local/benchmarks/ (see xilin/benchmarks/README.md), or set {FLAG} = True (paid API calls).")
    meta = json.loads((B1 / m / "meta.json").read_text())
    print(m, "| training", meta["training_start"], "..", meta["training_end"], "| model", meta["model"],
          "| steps", meta["steps"], "| tokens", meta["usage"], "|", meta["seconds"], "s")

## 2. The chain of thought

Rendered from the log with step ids (`[S012]`), the same text the judge reads. Each agent's prompts appear once.
`print(render(traj))` prints everything.

In [ ]:
MONTH = "2024-07"
traj, meta = load_trajectory(MONTH)
txt = render(traj)
print(f"{len(traj)} events, {len(txt):,} characters\n")
# skip the long prompt blocks for the preview: show the agents' own steps
steps = [l for l in txt.split("\n") if l.startswith("[S") or l.startswith("=====") or l.startswith("      RESULT")]
print("\n".join(s[:260] for s in steps[:80]))

In [ ]:
print(open(B1 / MONTH / f"systematic_strategy_{pd.Timestamp(meta['eval_date']).strftime('%m_%Y')}_gpt4o.txt").read())

## 3. Deterministic audit (no LLM)

Facts a script can establish. The judge receives them and is told to trust them.

In [ ]:
A = {m: audit(*load_trajectory(m)) for m in MONTHS}
rows = []
for m, a in A.items():
    rows.append({"month": m, "LLM calls": sum(a["llm_calls_by_agent"].values()), "tool calls": sum(a["tool_calls_by_tool"].values()),
                 "tool errors": len(a["tool_errors"]), "possible look-ahead": len(a["possible_look_ahead"]),
                 "training queries using ret_fwd_1": len(a["training_queries_using_ret_fwd_1"]),
                 "final conditions copied from prompt examples": f'{len(a["final_rule_conditions_copied_from_prompt_examples"])} '
                                                                 f'({a["share_of_final_conditions_copied"]:.0%})',
                 "final rule on decision date (BUY/SELL/overlap)": "{BUY}/{SELL}/{overlap}".format(**a["final_rule_on_decision_date"])
                     if not a["final_rule_on_decision_date"]["error"] else "NOT EXECUTABLE: " + a["final_rule_on_decision_date"]["error"]})
display(pd.DataFrame(rows).set_index("month").T)
for m, a in A.items():
    display(Markdown(f"**{m}** final rule: BUY `{a['final_rule']['BUY']}` | SELL `{a['final_rule']['SELL']}`  \n"
                     f"copied from prompt examples: {a['final_rule_conditions_copied_from_prompt_examples']}  \n"
                     f"tool calls: {a['tool_calls_by_tool']}  \nlook-ahead flags: {a['possible_look_ahead'] or 'none'}"))

## 4. LLM judge (Claude Opus 5.5)

The rubric (0 = missing or incorrect, 1 = partially supported, 2 = fully supported):

In [ ]:
display(pd.DataFrame(RUBRIC, columns=["id", "item", "anchors"]).set_index("id"))

In [ ]:
J = {}
for m in MONTHS:
    p = B1 / m / "judge.json"
    if RUN_JUDGE:
        print("judging", m); judge(m)
    elif not p.exists():
        FLAG = "RUN_JUDGE"
        raise FileNotFoundError(f"No saved run in {B1 / m}. Unzip the shared benchmark runs into "
                                f"data_local/benchmarks/ (see xilin/benchmarks/README.md), or set {FLAG} = True (paid API calls).")
    J[m] = json.loads(p.read_text())
    print(f"{m}: judge {J[m]['judge_model']}, {J[m]['judge_sql_checks']} SQL re-checks, ~${cost_usd(J[m]['usage']):.2f}")

In [ ]:
name = {i: n for i, n, _ in RUBRIC}
for m, j in J.items():
    display(Markdown(f"### {m}: overall **{j['overall_score']}/2**, tools helpful **{j['tools_helpful_score']}/2**\n\n{j['overall_summary']}"))
    t = pd.DataFrame(j["items"]).assign(item=lambda d: d.id.map(name))[["id", "item", "score", "justification", "evidence_steps", "verification"]]
    display(t.set_index("id"))
    display(Markdown("**Tools:** " + j["tools_used_summary"] + "\n\n**Were they helpful?** " + j["tools_helpful_explanation"]))
    display(Markdown("**Red flags:** " + "; ".join(j["red_flags"] or ["none"])))

In [ ]:
# What the judge re-checked with SQL (its own tool calls), so a human can audit the judge too
for m in MONTHS:
    log = [json.loads(l) for l in open(B1 / m / "judge_log.jsonl")]
    q = [r for r in log if r.get("event") == "tool"]
    display(Markdown(f"**{m}**: {len(q)} judge queries"))
    display(pd.DataFrame([{"query": r["query"], "result": r["output"][:200]} for r in q]))

## 5. Human check

Fill in the reviewer columns (0/1/2) after reading the cited steps. Agreement between the judge and the reviewers
(and between two reviewers) is the reliability check Capponi asked for. The table is saved to
`xilin/benchmarks/results/` (scores and text only, no licensed data).

In [ ]:
RESULTS.mkdir(parents=True, exist_ok=True)
sheets = []
for m, j in J.items():
    s = pd.DataFrame(j["items"]).assign(month=m, item=lambda d: d.id.map(name))
    s = s[["month", "id", "item", "score", "justification", "evidence_steps"]].rename(columns={"score": "judge_score"})
    s["reviewer1_score"] = ""; s["reviewer2_score"] = ""; s["notes"] = ""
    sheets.append(s)
sheet = pd.concat(sheets)
sheet.to_csv(RESULTS / "b1_rubric_review_sheet.csv", index=False)
(RESULTS / "b1_judge_scores.json").write_text(json.dumps(
    {m: {k: j[k] for k in ("overall_score", "tools_helpful_score", "overall_summary", "items", "red_flags", "judge_model")} for m, j in J.items()}, indent=2))
display(sheet.pivot(index=["id", "item"], columns="month", values="judge_score"))
print("saved:", RESULTS / "b1_rubric_review_sheet.csv")

## 6. Outcome (revealed only after scoring)

Realized next-month excess return of the names selected by each month's final rule, applied to the decision-date
cross-section. Two months is far too few to relate process scores to performance; this only shows the pipeline end to end.

In [ ]:
display(pd.DataFrame([outcome(m) for m in MONTHS]).set_index("eval_date"))

## Notes for discussion
- **Fidelity.** Prompts, agents, tools and loop are using the repo's notebook code, executed unchanged except for the LLM, base URL and rate limit.
- **Temperature 0 is not deterministic** for GPT-4o. A second run can differ; for the paper, use ≥3 runs per month and report score dispersion.
- **The judge is blind to outcomes** (masked SQL, no returns in the prompt), so the rubric is a stage-1 score in Prof. Capponi's sense.
- **Cross-family judge.** Currently I am using Claude to grade GPT-4o, so the LLM judge is not grading its own family's reasoning to avoid self-preference bias. Running a second judge (Gemini or alike) and reporting agreement is a cheap next step.